In [1]:
# deter. model- h1 (target def) + h2 (inspector noise) + screening
# same data pipeline as bridge_v4 + deterioration_v5, then  4 targets on top
# run from Documents/bridge_project, raw nbi files r in Downloads
# from bridge_v4 + deterioration_v5 nd rerun here, so the numbers come from this run
import os
import json
import warnings
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')
# print numpy numbers as plain #s (numpy 2 shows np.float64(...) otherwise)
np.set_printoptions(legacy='1.25')

UP = os.path.expanduser("~/Downloads")
OUT = "data"
os.makedirs(OUT, exist_ok=True)

# primary records only (005A = 1), the 2 records r basically empty copies
def load_nbi(path):
    d = pd.read_csv(path, encoding='latin-1', low_memory=False, on_bad_lines='skip')
    d = d[d['RECORD_TYPE_005A'].astype(str).str.strip() == '1'].copy()
    return d

# 2025 -- pull pr tx la out of the all states file, big so read in chunks
state_codes = {72: 'PR', 48: 'TX', 22: 'LA'}
chunks = []
for chunk in pd.read_csv(f"{UP}/2025HwyBridgesDelimitedAllStates.txt", encoding='latin-1',
                         low_memory=False, on_bad_lines='skip', chunksize=200_000):
    sub = chunk[chunk['STATE_CODE_001'].isin(state_codes.keys())]
    if len(sub):
        chunks.append(sub)
df25 = pd.concat(chunks, ignore_index=True)
df25['ST'] = df25['STATE_CODE_001'].map(state_codes)
df25 = df25[df25['RECORD_TYPE_005A'].astype(str).str.strip() == '1'].copy()
print("2025:", df25['ST'].value_counts().to_dict(), "total", len(df25))
df25.to_pickle(f"{OUT}/nbi_2025.pkl")

# 2020 snapshot
df20_pr = load_nbi(f"{UP}/NBI_PR20.txt"); df20_pr['ST'] = 'PR'
df20_tx = load_nbi(f"{UP}/NBI_TX20.txt"); df20_tx['ST'] = 'TX'
df20_la = load_nbi(f"{UP}/NBI_LA20.txt"); df20_la['ST'] = 'LA'
df20 = pd.concat([df20_pr, df20_tx, df20_la], ignore_index=True)
print("2020: PR", len(df20_pr), "TX", len(df20_tx), "LA", len(df20_la))
df20.to_pickle(f"{OUT}/nbi_2020.pkl")

# 2016 snapshot -- pr nd tx only, no la16 file
df16_pr = load_nbi(f"{UP}/NBI_PR16.txt"); df16_pr['ST'] = 'PR'
df16_tx = load_nbi(f"{UP}/NBI_TX16.txt"); df16_tx['ST'] = 'TX'
df16 = pd.concat([df16_pr, df16_tx], ignore_index=True)
print("2016: PR", len(df16_pr), "TX", len(df16_tx))
df16.to_pickle(f"{OUT}/nbi_2016.pkl")

2025: {'TX': 56951, 'LA': 12684, 'PR': 2434} total 72069
2020: PR 2334 TX 54682 LA 12853
2016: PR 2308 TX 53488


In [2]:
# coords + the 2 panels (2020 -> 2025 nd 2016 -> 2020)
# nbi lat/lon r packed DDMMSSss / DDDMMSSss, garbage or negative values -> nan
def nbi_coord(v, is_lon=False):
    if pd.isna(v):
        return np.nan
    iv = int(v)
    if iv <= 0:
        return np.nan
    s = str(iv).zfill(9 if is_lon else 8)
    if len(s) > (9 if is_lon else 8):
        return np.nan
    try:
        if is_lon:
            deg, mn, sec, frac = int(s[0:3]), int(s[3:5]), int(s[5:7]), int(s[7:9])
        else:
            deg, mn, sec, frac = int(s[0:2]), int(s[2:4]), int(s[4:6]), int(s[6:8])
        val = deg + mn/60 + (sec + frac/100)/3600
        return -val if is_lon else val
    except Exception:
        return np.nan

# box per state, 1 big box let bad pr coords thru
STATE_BOUNDS = {
    'PR': {'lat': (17.4, 18.7), 'lon': (-67.6, -64.9)},
    'TX': {'lat': (25.5, 36.8), 'lon': (-106.8, -93.2)},
    'LA': {'lat': (28.7, 33.2), 'lon': (-94.2, -88.6)},
}

def add_coords(df):
    df = df.copy()
    df['lat'] = df['LAT_016'].apply(lambda v: nbi_coord(v, is_lon=False))
    df['lon'] = df['LONG_017'].apply(lambda v: nbi_coord(v, is_lon=True))
    bad = pd.Series(True, index=df.index)
    for st, b in STATE_BOUNDS.items():
        m = df['ST'] == st
        ok = df['lat'].between(*b['lat']) & df['lon'].between(*b['lon'])
        bad.loc[m] = ~ok.loc[m]
    print("  bad coords set to nan:", int(bad.sum()), "of", len(df))
    df.loc[bad, ['lat', 'lon']] = np.nan
    return df

COMP = ['DECK_COND_058', 'SUPERSTRUCTURE_COND_059', 'SUBSTRUCTURE_COND_060']
for df in (df25, df20, df16):
    for c in COMP:
        df[c] = pd.to_numeric(df[c], errors='coerce')
df25, df20, df16 = add_coords(df25), add_coords(df20), add_coords(df16)

# bridge key = state + structure number (structure numbers repeat across states)
for df in (df25, df20, df16):
    df['bkey'] = df['ST'] + '_' + df['STRUCTURE_NUMBER_008'].astype(str).str.strip()

key_feats = ['bkey', 'ST', 'lat', 'lon', 'YEAR_BUILT_027', 'ADT_029', 'PERCENT_ADT_TRUCK_109',
             'STRUCTURE_KIND_043A', 'SCOUR_CRITICAL_113', 'CHANNEL_COND_061', 'WATERWAY_EVAL_071',
             'DECK_COND_058', 'SUPERSTRUCTURE_COND_059', 'SUBSTRUCTURE_COND_060',
             'YEAR_RECONSTRUCTED_106', 'WORK_PROPOSED_075A']
key_out = ['bkey'] + COMP

# panel a: 2020 feats -> 2025 outcome
p20 = df20[[c for c in key_feats if c in df20.columns]].copy().add_prefix('f20_').rename(columns={'f20_bkey': 'bkey'})
o25 = df25[key_out].copy().add_prefix('y25_').rename(columns={'y25_bkey': 'bkey'})
panelA = p20.merge(o25, on='bkey', how='inner')
# panel b: 2016 feats -> 2020 outcome
p16 = df16[[c for c in key_feats if c in df16.columns]].copy().add_prefix('f16_').rename(columns={'f16_bkey': 'bkey'})
o20 = df20[key_out].copy().add_prefix('y20_').rename(columns={'y20_bkey': 'bkey'})
panelB = p16.merge(o20, on='bkey', how='inner')
print("panel a (2020->2025):", len(panelA), " panel b (2016->2020):", len(panelB))

  bad coords set to nan: 4 of 72069
  bad coords set to nan: 173 of 69869
  bad coords set to nan: 22 of 55796
panel a (2020->2025): 67329  panel b (2016->2020): 54513


In [3]:
# climate (daymet, 0.25 deg cells), distance to coast, maintenance history
clim = pd.read_csv(f"{OUT}/climate_v4_daymet.csv")
CLIM_COLS = [c for c in clim.columns if c.startswith('clim_')]
RES = 0.25

# hand made coastline pts (gulf + pr), not a real shapefile so its a rough proxy
COASTLINE = [
    (25.96, -97.15), (26.07, -97.17), (26.38, -97.22), (26.68, -97.30),
    (27.02, -97.38), (27.38, -97.28), (27.55, -97.22), (27.80, -97.05),
    (27.90, -97.07), (28.02, -97.05), (28.13, -96.90), (28.30, -96.72),
    (28.42, -96.62), (28.60, -96.40), (28.68, -96.20), (28.80, -95.95),
    (28.95, -95.60), (29.05, -95.30), (29.15, -95.10), (29.28, -94.90),
    (29.30, -94.80), (29.35, -94.72), (29.45, -94.68), (29.55, -94.60),
    (29.68, -94.40), (29.73, -94.05), (29.73, -93.87),
    (29.75, -93.60), (29.60, -93.30), (29.55, -93.00), (29.50, -92.70),
    (29.55, -92.30), (29.55, -92.00), (29.45, -91.70), (29.35, -91.40),
    (29.25, -91.20), (29.15, -91.00), (29.10, -90.80), (29.05, -90.60),
    (29.10, -90.40), (29.05, -90.10), (28.95, -89.95), (28.85, -89.55),
    (28.90, -89.40), (29.10, -89.25), (29.15, -89.10), (29.25, -89.10),
    (29.30, -89.30), (29.40, -89.35), (29.50, -89.30), (29.60, -89.35),
    (29.65, -89.45), (29.70, -89.55), (29.75, -89.60), (29.85, -89.65),
    (29.90, -89.60), (29.95, -89.55), (30.00, -89.50), (30.10, -89.50),
    (30.17, -89.50),
    (18.47, -67.24), (18.49, -67.10), (18.50, -66.90), (18.50, -66.60),
    (18.48, -66.30), (18.45, -66.10), (18.38, -65.90), (18.20, -65.75),
    (18.10, -65.80), (17.95, -66.00), (17.93, -66.30), (17.95, -66.60),
    (17.97, -66.90), (18.00, -67.10), (18.05, -67.20), (18.15, -67.25),
    (18.30, -67.25), (18.47, -67.24),
]
COAST_LAT = np.array([p[0] for p in COASTLINE])
COAST_LON = np.array([p[1] for p in COASTLINE])

def dist_to_coast(lat, lon):
    if pd.isna(lat) or pd.isna(lon):
        return np.nan
    la1, lo1 = np.radians(lat), np.radians(lon)
    la2, lo2 = np.radians(COAST_LAT), np.radians(COAST_LON)
    a = np.sin((la2 - la1)/2)**2 + np.cos(la1)*np.cos(la2)*np.sin((lo2 - lo1)/2)**2
    return float((2 * 6371.0 * np.arcsin(np.sqrt(a))).min())

def attach_climate(df, latcol, loncol, prefix):
    df = df.copy()
    key = pd.DataFrame({'gy': (df[latcol] / RES).round(), 'gx': (df[loncol] / RES).round()})
    merged = key.merge(clim[['gy', 'gx'] + CLIM_COLS], on=['gy', 'gx'], how='left')
    for c in CLIM_COLS:
        df[f'{prefix}_{c}'] = merged[c].values
    # cell centroid over water = no daymet data, use nearest cell that has it
    miss = df[f'{prefix}_{CLIM_COLS[0]}'].isna() & df[latcol].notna() & df[loncol].notna()
    if miss.sum():
        idx = [int(np.argmin((clim['lat'].values - la)**2 + (clim['lon'].values - lo)**2))
               for la, lo in zip(df.loc[miss, latcol], df.loc[miss, loncol])]
        for c in CLIM_COLS:
            df.loc[miss, f'{prefix}_{c}'] = clim[c].values[idx]
    print(f"  {prefix}: nearest cell fill {int(miss.sum())}, still no climate {int(df[f'{prefix}_{CLIM_COLS[0]}'].isna().sum())}")
    return df

def attach_maint(df, prefix, panel_year):
    df = df.copy()
    yr = pd.to_numeric(df[f'{prefix}_YEAR_RECONSTRUCTED_106'], errors='coerce').fillna(0)
    df[f'{prefix}_ever_reconstructed'] = (yr > 0).astype(int)
    built_age = panel_year - df[f'{prefix}_YEAR_BUILT_027']
    df[f'{prefix}_years_since_major_work'] = np.where(yr > 0, panel_year - yr, built_age).clip(min=0)
    work = df[f'{prefix}_WORK_PROPOSED_075A']
    df[f'{prefix}_work_proposed_flag'] = work.notna().astype(int)
    # 31 = replacement
    df[f'{prefix}_work_is_replacement'] = (work == 31).astype(int)
    return df

for name, latc, lonc, pre, yr in [('A', 'f20_lat', 'f20_lon', 'f20', 2020), ('B', 'f16_lat', 'f16_lon', 'f16', 2016)]:
    pnl = panelA if name == 'A' else panelB
    pnl = attach_climate(pnl, latc, lonc, pre)
    pnl[f'{pre}_dist_coast_km'] = [dist_to_coast(la, lo) for la, lo in zip(pnl[latc], pnl[lonc])]
    pnl = attach_maint(pnl, pre, yr)
    if name == 'A':
        panelA = pnl
    else:
        panelB = pnl
print("dist to coast by state (panel a):")
print(panelA.groupby('f20_ST')['f20_dist_coast_km'].describe()[['mean', 'min', 'max']])

  f20: nearest cell fill 5, still no climate 171
  f16: nearest cell fill 9, still no climate 22
dist to coast by state (panel a):
              mean       min          max
f20_ST                                   
LA      169.625745  2.190623   384.008479
PR       14.020271  0.370650    48.871793
TX      327.777830  0.824332  1063.007117


In [4]:
# stack both panels into 1 frame, same col names
def strip(df, fpre, ypre, panel_name, feat_year):
    ren = {}
    for c in df.columns:
        if c.startswith(fpre + '_'):
            ren[c] = c[len(fpre) + 1:]
        elif c.startswith(ypre + '_'):
            ren[c] = 'y_' + c[len(ypre) + 1:]
    df = df.rename(columns=ren)
    df['panel'] = panel_name
    df['feat_year'] = feat_year
    return df

pn = pd.concat([strip(panelA, 'f20', 'y25', 'A_2020_2025', 2020),
                strip(panelB, 'f16', 'y20', 'B_2016_2020', 2016)], ignore_index=True, sort=False)
print("stacked rows:", len(pn))

# N = not over water, U = unknown -> nan -> median
for c in ['SCOUR_CRITICAL_113', 'CHANNEL_COND_061', 'WATERWAY_EVAL_071']:
    pn[c] = pd.to_numeric(pn[c], errors='coerce')
    pn[c] = pn[c].fillna(pn[c].median())

# material one hot
mats = pd.get_dummies(pn['STRUCTURE_KIND_043A'].astype(str), prefix='mat').astype(int)
pn = pd.concat([pn, mats], axis=1)

# any component dropped (the original 1 pt target), nan if a rating is missing
y_cols = ['y_' + c for c in COMP]
missing = pn[y_cols].isna().any(axis=1) | pn[COMP].isna().any(axis=1)
pn['min_cond_now'] = pn[COMP].min(axis=1)
deter = ((pn['y_DECK_COND_058'] < pn['DECK_COND_058']) |
         (pn['y_SUPERSTRUCTURE_COND_059'] < pn['SUPERSTRUCTURE_COND_059']) |
         (pn['y_SUBSTRUCTURE_COND_060'] < pn['SUBSTRUCTURE_COND_060']))
pn['deterB'] = deter.astype('Int64')
pn.loc[missing, 'deterB'] = pd.NA
pn['bridge_age'] = pn['feat_year'] - pn['YEAR_BUILT_027']

base_p = ['bridge_age', 'ADT_029', 'PERCENT_ADT_TRUCK_109', 'SCOUR_CRITICAL_113',
          'CHANNEL_COND_061', 'WATERWAY_EVAL_071', 'lat', 'lon'] + list(mats.columns)
clim_p = [c for c in pn.columns if c.startswith('clim_')]
coast_p = ['dist_coast_km']
maint_p = ['ever_reconstructed', 'years_since_major_work', 'work_proposed_flag', 'work_is_replacement']
lag_p = COMP + ['min_cond_now']
featsP_full = base_p + clim_p + coast_p + maint_p + lag_p
print("base", len(base_p), "clim", len(clim_p), "coast", len(coast_p), "maint", len(maint_p), "lag", len(lag_p),
      "-> featsP_full", len(featsP_full))

stacked rows: 121842
base 18 clim 14 coast 1 maint 4 lag 4 -> featsP_full 41


In [5]:
# extra feature year nbi fields + engineered feats + own 2016->2020 history
# only stuff known at the feature year, nothing from the outcome year except the target
def load_raw(year):
    d = pd.read_pickle(f"{OUT}/nbi_{year}.pkl")
    d['bkey'] = d['ST'] + '_' + d['STRUCTURE_NUMBER_008'].astype(str).str.strip()
    return d.drop_duplicates('bkey')

NUMERIC = ['DETOUR_KILOS_019', 'TRAFFIC_LANES_ON_028A', 'TRAFFIC_LANES_UND_028B', 'YEAR_ADT_030',
           'APPR_WIDTH_MT_032', 'DEGREES_SKEW_034', 'MAIN_UNIT_SPANS_045', 'APPR_SPANS_046',
           'HORR_CLR_MT_047', 'MAX_SPAN_LEN_MT_048', 'STRUCTURE_LEN_MT_049', 'ROADWAY_WIDTH_MT_051',
           'DECK_WIDTH_MT_052', 'VERT_CLR_UND_054B', 'OPERATING_RATING_064', 'INVENTORY_RATING_066',
           'INSPECT_FREQ_MONTHS_091', 'BRIDGE_IMP_COST_094', 'ROADWAY_IMP_COST_095',
           'TOTAL_IMP_COST_096', 'YEAR_OF_IMP_097', 'IMP_LEN_MT_076', 'FUTURE_ADT_114',
           'YEAR_OF_FUTURE_ADT_115', 'COUNTY_CODE_003', 'HIGHWAY_DISTRICT_002']
ORDINAL = ['CULVERT_COND_062', 'STRUCTURAL_EVAL_067', 'DECK_GEOMETRY_EVAL_068',
           'UNDCLRENCE_EVAL_069', 'POSTING_EVAL_070', 'APPR_ROAD_EVAL_072']
CATEG = ['MAINTENANCE_021', 'OWNER_022', 'FUNCTIONAL_CLASS_026', 'DESIGN_LOAD_031',
         'OPEN_CLOSED_POSTED_041', 'SERVICE_ON_042A', 'SERVICE_UND_042B',
         'STRUCTURE_TYPE_043B', 'APPR_KIND_044A', 'DECK_STRUCTURE_TYPE_107',
         'SURFACE_TYPE_108A', 'MEMBRANE_TYPE_108B', 'DECK_PROTECTION_108C',
         'HISTORY_037', 'PIER_PROTECTION_111', 'HIGHWAY_SYSTEM_104', 'TOLL_020',
         'WORK_DONE_BY_075B', 'MEDIAN_CODE_033', 'NAVIGATION_038', 'TRAFFIC_DIRECTION_102']
FLAGS = ['FRACTURE_092A', 'UNDWATER_LOOK_SEE_092B', 'SPEC_INSPECT_092C']  # 'Y12' or 'N'

def extract(raw, feat_year):
    f = pd.DataFrame({'bkey': raw['bkey']})
    for c in NUMERIC:
        f['n_' + c] = pd.to_numeric(raw[c], errors='coerce')
    for c in ORDINAL:
        f['o_' + c] = pd.to_numeric(raw[c].astype(str).str.strip(), errors='coerce')
    for c in CATEG:
        f['c_' + c] = raw[c].astype(str).str.strip()
    for c in FLAGS:
        s = raw[c].astype(str).str.strip()
        f['f_' + c] = s.str.startswith('Y').astype(int)
        f['fm_' + c] = pd.to_numeric(s.str[1:], errors='coerce')
    # months since last inspection at the snapshot, date is MMYY
    v = pd.to_numeric(raw['DATE_OF_INSPECT_090'], errors='coerce')
    mm, yy = v // 100, v % 100
    yr = np.where(yy > 50, 1900 + yy, 2000 + yy)
    f['insp_months_before_snapshot'] = (feat_year * 12 + 12) - (yr * 12 + mm)
    return f

ext = []
for year, panel in [(2020, 'A_2020_2025'), (2016, 'B_2016_2020')]:
    e = extract(load_raw(year), year)
    e['panel'] = panel
    ext.append(e)
ext = pd.concat(ext, ignore_index=True)
# categorical -> int codes, 1 mapping for both panels
for c in [x for x in ext.columns if x.startswith('c_')]:
    ext[c] = pd.Categorical(ext[c]).codes.astype(float)
    ext.loc[ext[c] < 0, c] = np.nan
pn = pn.merge(ext, on=['bkey', 'panel'], how='left')
raw_cols = [c for c in ext.columns if c not in ('bkey', 'panel')]

# engineered -- condition spread, DR = (9-CR)/age per component (O&O), traffic, peers
def add_cond_feats(df):
    C = df[COMP]
    df['min_cond_now'] = C.min(axis=1)
    df['cond_max'] = C.max(axis=1)
    df['cond_mean'] = C.mean(axis=1)
    df['cond_range'] = df['cond_max'] - df['min_cond_now']
    df['n_comp_at_min'] = C.eq(df['min_cond_now'], axis=0).sum(axis=1)
    df['n_comp_ge8'] = (C >= 8).sum(axis=1)
    df['deck_minus_sub'] = df['DECK_COND_058'] - df['SUBSTRUCTURE_COND_060']
    df['super_minus_sub'] = df['SUPERSTRUCTURE_COND_059'] - df['SUBSTRUCTURE_COND_060']
    age = df['bridge_age'].clip(lower=1)
    eff_age = df['years_since_major_work'].clip(lower=1)
    for c, s in zip(COMP, ['deck', 'super', 'sub']):
        df[f'dr_{s}'] = (9 - df[c]) / age
        df[f'dr_{s}_since_work'] = (9 - df[c]) / eff_age
    # condition vs same age bridges nd vs neighbors (feature year only, no labels)
    age_bin = pd.cut(df['bridge_age'], bins=[-1, 5, 10, 15, 20, 30, 40, 50, 60, 80, 200])
    df['cond_vs_age_peers'] = df['min_cond_now'] - df.groupby(['panel', age_bin], observed=True)['min_cond_now'].transform('mean')
    for key, nm in [('gcell', 'cell'), ('cty', 'cty')]:
        g = df.groupby(['panel', key])['min_cond_now']
        df[f'{nm}_mean_cond'] = g.transform('mean')
        df[f'{nm}_n_bridges'] = g.transform('size')
        df[f'cond_vs_{nm}'] = df['min_cond_now'] - df[f'{nm}_mean_cond']
    return df

pn['gcell'] = (pn['lat'] / 0.25).round().astype('Int64').astype(str) + '_' + (pn['lon'] / 0.25).round().astype('Int64').astype(str)
pn['cty'] = pn['ST'] + '_' + pn['n_COUNTY_CODE_003'].astype('Int64').astype(str)
pn['window_years'] = np.where(pn['panel'] == 'A_2020_2025', 5, 4)
pn['truck_adt'] = pn['ADT_029'] * pn['PERCENT_ADT_TRUCK_109'] / 100
pn['adt_per_lane'] = pn['ADT_029'] / pn['n_TRAFFIC_LANES_ON_028A'].replace(0, np.nan)
pn['deck_area'] = pn['n_STRUCTURE_LEN_MT_049'] * pn['n_DECK_WIDTH_MT_052']
pn['adt_growth'] = pn['n_FUTURE_ADT_114'] / pn['ADT_029'].replace(0, np.nan)
pn = add_cond_feats(pn)

# own history: 2016->2020 change, only for panel a rows (pr + tx)
r16 = load_raw(2016)[['bkey'] + COMP].copy()
for c in COMP:
    r16[c] = pd.to_numeric(r16[c], errors='coerce')
r16 = r16.rename(columns={c: 'p16_' + c for c in COMP})
pn = pn.merge(r16, on='bkey', how='left')
isA = pn['panel'] == 'A_2020_2025'
for c, s in zip(COMP, ['deck', 'super', 'sub']):
    pn[f'hist_delta_{s}'] = np.where(isA, pn[c] - pn['p16_' + c], np.nan)
hd = pn[['hist_delta_deck', 'hist_delta_super', 'hist_delta_sub']]
pn['hist_any_drop'] = np.where(isA & hd.notna().all(axis=1), (hd < 0).any(axis=1).astype(float), np.nan)
pn['hist_any_rise'] = np.where(isA & hd.notna().all(axis=1), (hd > 0).any(axis=1).astype(float), np.nan)
pn['hist_sum_delta'] = np.where(isA, hd.sum(axis=1, min_count=3), np.nan)
pn['hist_available'] = pn['hist_any_drop'].notna().astype(int)
pn = pn.drop(columns=['p16_' + c for c in COMP])

eng_cols = ['cond_max', 'cond_mean', 'cond_range', 'n_comp_at_min', 'n_comp_ge8', 'deck_minus_sub',
            'super_minus_sub', 'dr_deck', 'dr_super', 'dr_sub', 'dr_deck_since_work',
            'dr_super_since_work', 'dr_sub_since_work', 'window_years', 'truck_adt', 'adt_per_lane',
            'deck_area', 'adt_growth', 'cond_vs_age_peers', 'cell_mean_cond', 'cell_n_bridges',
            'cond_vs_cell', 'cty_mean_cond', 'cty_n_bridges', 'cond_vs_cty']
hist_cols = ['hist_delta_deck', 'hist_delta_super', 'hist_delta_sub', 'hist_any_drop',
             'hist_any_rise', 'hist_sum_delta', 'hist_available']
FEATS = featsP_full + eng_cols + raw_cols + hist_cols
print("raw", len(raw_cols), "engineered", len(eng_cols), "history", len(hist_cols), "-> total feats", len(FEATS))

raw 60 engineered 25 history 7 -> total feats 133


In [6]:
# modeling rows (same 65,204 as v5) + the 4 deterioration targets
ROWS = pn.dropna(subset=featsP_full + ['deterB']).reset_index(drop=True)
print("modeling rows:", len(ROWS), " by panel:", ROWS['panel'].value_counts().to_dict())

y_cols = ['y_' + c for c in COMP]
ROWS['y_min'] = ROWS[y_cols].min(axis=1)
drops = (ROWS[COMP].values - ROWS[y_cols].values).clip(min=0)

T = {}
# 1. any component drops 1+ pt (the old target)
T['any_drop1'] = ROWS['deterB'].astype(int).values
# 2. weakest component drops 1+ pt
T['min_drop1'] = (ROWS['y_min'] < ROWS['min_cond_now']).astype(int).values
# 3. 2+ total pts lost across the 3 components
T['total_drop2'] = (drops.sum(axis=1) >= 2).astype(int)
# 4. enters federal poor: min rating 5+ -> below 5 (only bridges that start at 5+ can enter)
ELIG = (ROWS['min_cond_now'] >= 5).values
T['enters_poor'] = (ROWS['y_min'] < 5).astype(int).values

for k, v in T.items():
    m = ELIG if k == 'enters_poor' else np.ones(len(ROWS), bool)
    print(f"{k}: rows {int(m.sum())}  base rate {round(v[m].mean(), 4)}  events {int(v[m].sum())}")

modeling rows: 65204  by panel: {'A_2020_2025': 40882, 'B_2016_2020': 24322}
any_drop1: rows 65204  base rate 0.3084  events 20111
min_drop1: rows 65204  base rate 0.1537  events 10024
total_drop2: rows 65204  base rate 0.101  events 6585
enters_poor: rows 63627  base rate 0.0137  events 873


In [7]:
# rf + cv harness
# 300 trees, entropy, min 2 per leaf, balanced class weights
# folds grouped by bridge so the same bridge is never in train nd test
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_auc_score

X_ALL = ROWS[FEATS].astype(float).values
BKEY = ROWS['bkey'].values
CTY = ROWS['cty'].values
isB = (ROWS['panel'] == 'B_2016_2020').values
# forward in time cant use the own history cols (all nan in 2016) or window length
FWD_FEATS = [f for f in FEATS if f not in hist_cols and f != 'window_years']
X_FWD = ROWS[FWD_FEATS].astype(float).values

def make_rf():
    return RandomForestClassifier(n_estimators=300, criterion='entropy', min_samples_leaf=2,
                                  max_features='sqrt', class_weight='balanced', n_jobs=-1, random_state=42)

def run_cv(X, y, groups, seed=7):
    oof = np.zeros(len(y))
    aucs = []
    for tr, te in StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed).split(X, y, groups):
        m = make_rf().fit(X[tr], y[tr])
        oof[te] = m.predict_proba(X[te])[:, 1]
        aucs.append(roc_auc_score(y[te], oof[te]))
    return oof, np.array(aucs)

def run_forward(X, y, mask):
    # train on every 2016->2020 window, test on every 2020->2025 window
    tr = np.where(isB & mask)[0]
    te = np.where(~isB & mask)[0]
    p = make_rf().fit(X[tr], y[tr]).predict_proba(X[te])[:, 1]
    return roc_auc_score(y[te], p), p, te

RES_OUT = {}

In [8]:
# H1 -- same feats, model, folds. ONLY the target changes
OOF = {}
for k in ['any_drop1', 'min_drop1', 'total_drop2', 'enters_poor']:
    m = ELIG if k == 'enters_poor' else np.ones(len(ROWS), bool)
    oof, aucs = run_cv(X_ALL[m], T[k][m], BKEY[m])
    OOF[k] = (oof, m)
    pooled = roc_auc_score(T[k][m], oof)
    RES_OUT[f'random_{k}'] = {'auc_mean': round(aucs.mean(), 4), 'auc_sd': round(aucs.std(), 4), 'auc_pooled': round(pooled, 4)}
    print(f"{k}: AUC {round(aucs.mean(), 4)} ± {round(aucs.std(), 4)}  (pooled oof {round(pooled, 4)})")

any_drop1: AUC 0.7846 ± 0.0039  (pooled oof 0.7846)
min_drop1: AUC 0.8188 ± 0.0031  (pooled oof 0.8188)
total_drop2: AUC 0.8327 ± 0.0069  (pooled oof 0.8326)
enters_poor: AUC 0.92 ± 0.0117  (pooled oof 0.9194)


In [9]:
# rarity check -- is enters poor just easier bc its rare?
# control 1: any_drop1 positives thinned to the same rate as enters poor
# control 2: same rate but the positives r picked at random (no real signal)
p_ep = T['enters_poor'][ELIG].mean()
y1 = T['any_drop1']
neg = np.where(y1 == 0)[0]
pos = np.where(y1 == 1)[0]
c1, c2 = [], []
for d in range(5):
    rng = np.random.default_rng(d)
    k = int(round(p_ep * len(neg) / (1 - p_ep)))
    keep = np.concatenate([neg, rng.choice(pos, k, replace=False)])
    _, a = run_cv(X_ALL[keep], y1[keep], BKEY[keep])
    c1.append(a.mean())
    yr = np.zeros(len(ROWS), int)
    yr[rng.choice(pos, int(round(p_ep * len(ROWS))), replace=False)] = 1
    _, a = run_cv(X_ALL, yr, BKEY)
    c2.append(a.mean())
    print(f"draw {d}: control 1 {round(c1[-1], 4)}  control 2 {round(c2[-1], 4)}")
RES_OUT['rarity'] = {'base_rate': round(p_ep, 4), 'control1_mean': round(np.mean(c1), 4), 'control1_sd': round(np.std(c1), 4),
                     'control2_mean': round(np.mean(c2), 4), 'control2_sd': round(np.std(c2), 4)}
print("control 1 (thinned any drop):", round(np.mean(c1), 4), "| control 2 (random positives):", round(np.mean(c2), 4))
print("real enters poor:", RES_OUT['random_enters_poor']['auc_mean'])

draw 0: control 1 0.6797  control 2 0.5922
draw 1: control 1 0.6978  control 2 0.615
draw 2: control 1 0.6979  control 2 0.5853
draw 3: control 1 0.6839  control 2 0.6041
draw 4: control 1 0.6842  control 2 0.5743
control 1 (thinned any drop): 0.6887 | control 2 (random positives): 0.5942
real enters poor: 0.92


In [10]:
# validation on unseen data
# unseen county = folds grouped by county, forward = train 2016-20 test 2020-25
FWD_P = {}
for k in ['any_drop1', 'total_drop2', 'enters_poor']:
    m = ELIG if k == 'enters_poor' else np.ones(len(ROWS), bool)
    oof_c, a_c = run_cv(X_ALL[m], T[k][m], CTY[m])
    auc_f, p_f, te_f = run_forward(X_FWD, T[k], m)
    FWD_P[k] = (p_f, te_f)
    OOF[k + '_county'] = (oof_c, m)
    r = RES_OUT[f'random_{k}']['auc_mean']
    RES_OUT[f'unseen_{k}'] = {'county': round(a_c.mean(), 4), 'forward': round(auc_f, 4), 'gap': round(r - auc_f, 4),
                             'forward_n': int(len(te_f))}
    print(f"{k}: random {r}  unseen county {round(a_c.mean(), 4)}  forward {round(auc_f, 4)}  gap {round(r - auc_f, 4)}  (forward test n={len(te_f)})")

any_drop1: random 0.7846  unseen county 0.7257  forward 0.6626  gap 0.122  (forward test n=40882)
total_drop2: random 0.8327  unseen county 0.7839  forward 0.7124  gap 0.1203  (forward test n=40882)
enters_poor: random 0.92  unseen county 0.8952  forward 0.8798  gap 0.0402  (forward test n=39657)


In [14]:
# screening value -- if u only inspect the riskiest 10%, how many enters poor bridges do u catch
def top10(y, p):
    n = int(np.ceil(0.10 * len(p)))
    top = np.argsort(-p)[:n]
    cap = y[top].sum() / y.sum()
    prec = y[top].mean()
    return round(cap, 4), round(prec, 4), round(cap / 0.10, 2)

y_ep = T['enters_poor']
oof, m = OOF['enters_poor']
print("random split:", top10(y_ep[m], oof))
oof_c, m = OOF['enters_poor_county']
print("unseen county:", top10(y_ep[m], oof_c))
p_f, te_f = FWD_P['enters_poor']
print("forward in time:", top10(y_ep[te_f], p_f))
RES_OUT['screening'] = {'random': top10(y_ep[OOF['enters_poor'][1]], OOF['enters_poor'][0]),
                        'county': top10(y_ep[m], oof_c), 'forward': top10(y_ep[te_f], p_f)}
# (capture, precision, lift)

# 95% ci for enters poor auc, bootstrap the out of fold preds
from sklearn.utils import resample
oof, m = OOF['enters_poor']
yy = y_ep[m]
boot = []
for i in range(1000):
    idx = resample(np.arange(len(yy)), random_state=i)
    if yy[idx].sum() > 0:
        boot.append(roc_auc_score(yy[idx], oof[idx]))
boot = np.array(boot)
RES_OUT['enters_poor_ci'] = [round(np.percentile(boot, 2.5), 4), round(np.percentile(boot, 97.5), 4)]
print("enters poor AUC 95% CI:", RES_OUT['enters_poor_ci'])

random split: (0.7423, 0.1018, 7.42)
unseen county: (0.6838, 0.0938, 6.84)
forward in time: (0.6066, 0.0925, 6.07)
enters poor AUC 95% CI: [0.9105, 0.9282]


In [15]:
# H2 -- inspector noise
# moore et al 2001: 68% of ratings within +-1 pt -> sd about 1 pt per rating
# add N(0, sigma) to the feature year AND outcome year ratings, round, clip 0-9,
# rebuild the condition feats nd the targets, then rerun the same model
def noisy_copy(sigma, seed=0):
    rng = np.random.default_rng(seed)
    d = ROWS.copy()
    for c in COMP:
        d[c] = np.clip(np.round(d[c] + rng.normal(0, sigma, len(d))), 0, 9)
        d['y_' + c] = np.clip(np.round(d['y_' + c] + rng.normal(0, sigma, len(d))), 0, 9)
    d = add_cond_feats(d)
    y_any = (d[[ 'y_' + c for c in COMP]].values < d[COMP].values).any(axis=1).astype(int)
    elig = (d['min_cond_now'] >= 5).values
    y_ep = (d[['y_' + c for c in COMP]].min(axis=1) < 5).astype(int).values
    return d, y_any, y_ep, elig

NOISE = {}
for sigma in [0, 0.5, 1.0, 1.5]:
    if sigma == 0:
        # sigma 0 = wiring check, should match the H1 numbers exactly
        d, y_any, y_ep_n, elig = ROWS, T['any_drop1'], T['enters_poor'], ELIG
    else:
        d, y_any, y_ep_n, elig = noisy_copy(sigma)
    Xn = d[FEATS].astype(float).values
    Xf = d[FWD_FEATS].astype(float).values
    out = {}
    for name, y, m in [('enters_poor', y_ep_n, elig), ('any_drop1', y_any, np.ones(len(d), bool))]:
        _, a = run_cv(Xn[m], y[m], BKEY[m])
        auc_f, _, _ = run_forward(Xf, y, m)
        out[name] = {'base_rate': round(y[m].mean(), 4), 'random': round(a.mean(), 4), 'forward': round(auc_f, 4),
                     'gap': round(a.mean() - auc_f, 4)}
    NOISE[sigma] = out
    print(f"sigma {sigma}: enters poor {out['enters_poor']}  | any drop {out['any_drop1']}")
RES_OUT['noise'] = {str(k): v for k, v in NOISE.items()}

sigma 0: enters poor {'base_rate': 0.0137, 'random': 0.92, 'forward': 0.8798, 'gap': 0.0402}  | any drop {'base_rate': 0.3084, 'random': 0.7846, 'forward': 0.6626, 'gap': 0.122}
sigma 0.5: enters poor {'base_rate': 0.0349, 'random': 0.8865, 'forward': 0.864, 'gap': 0.0225}  | any drop {'base_rate': 0.6563, 'random': 0.7468, 'forward': 0.7318, 'gap': 0.0149}
sigma 1.0: enters poor {'base_rate': 0.1061, 'random': 0.7601, 'forward': 0.7484, 'gap': 0.0117}  | any drop {'base_rate': 0.7748, 'random': 0.7641, 'forward': 0.7538, 'gap': 0.0103}
sigma 1.5: enters poor {'base_rate': 0.2335, 'random': 0.6638, 'forward': 0.6501, 'gap': 0.0137}  | any drop {'base_rate': 0.8048, 'random': 0.7685, 'forward': 0.7633, 'gap': 0.0052}


In [16]:
# save everything nd print the summary
json.dump(RES_OUT, open(f"{OUT}/deterioration_results.json", 'w'), indent=2)
rows = []
for k in ['any_drop1', 'min_drop1', 'total_drop2', 'enters_poor']:
    u = RES_OUT.get(f'unseen_{k}', {})
    rows.append([k, RES_OUT[f'random_{k}']['auc_mean'], u.get('county', '-'), u.get('forward', '-'), u.get('gap', '-')])
print(pd.DataFrame(rows, columns=['target', 'random split', 'unseen county', 'forward in time', 'gap']).to_string(index=False))
print("\nrarity controls:", RES_OUT['rarity'])
print("screening (capture, precision, lift):", RES_OUT['screening'])
print("noise (enters poor random auc):", {k: v['enters_poor']['random'] for k, v in RES_OUT['noise'].items()})

     target  random split unseen county forward in time     gap
  any_drop1        0.7846        0.7257          0.6626   0.122
  min_drop1        0.8188             -               -       -
total_drop2        0.8327        0.7839          0.7124  0.1203
enters_poor        0.9200        0.8952          0.8798  0.0402

rarity controls: {'base_rate': 0.0137, 'control1_mean': 0.6887, 'control1_sd': 0.0076, 'control2_mean': 0.5942, 'control2_sd': 0.0142}
screening (capture, precision, lift): {'random': (0.7423, 0.1018, 7.42), 'county': (0.6838, 0.0938, 6.84), 'forward': (0.6066, 0.0925, 6.07)}
noise (enters poor random auc): {'0': 0.92, '0.5': 0.8865, '1.0': 0.7601, '1.5': 0.6638}
